# Concatenate Many Spectractor Results from Corentin Runs

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-13
- last update : 2026-11-13 : fixed output path (missing `../` prefix in `mergedextractedfilesdict`), added sanity checks, output directory creation and safe writing

Merge the AuxTel atmospheric-parameter tables (Spectractor results, one `.npy` file per run/month) into a single table sorted by `DATE-OBS`.
The merge composition and all file paths are defined in `BUTLER00_parameters.py`.

In [ ]:
from pathlib import Path
import pprint

import numpy as np
import pandas as pd

In [ ]:
# CHECK THE CONFIG IN BUTLER00_parameters.py BEFORE RUNNING !!!
from BUTLER00_parameters import *

DumpConfig()

## Configuration

### Choose Composition

In [ ]:
# Composition to merge: must be a key of both `merge_composition` and `mergedextractedfilesdict`
#compo_sel = 'run2026_v02b_cr_run2026_v02d_cr'
compo_sel = 'run2023-2026_JanToJul_cr'
#compo_sel = 'run2026_JanToJul_cr'

# Column used to identify a unique exposure (duplicates are dropped on this key)
KEY_COLUMN = 'DATE-OBS'

# If True, add a column 'RUN_TAG' recording which input run each row comes from.
# Default is False so that the output schema is unchanged.
ADD_SOURCE_COLUMN = False

# Fail early, with an explicit message, if the configuration is incomplete
for name, dico in [('merge_composition', merge_composition),
                   ('mergedextractedfilesdict', mergedextractedfilesdict)]:
    if compo_sel not in dico:
        raise KeyError(f"'{compo_sel}' not found in {name} (BUTLER00_parameters.py). Available: {list(dico)}")

tag_list = merge_composition[compo_sel]
unknown_tags = [tag for tag in tag_list if tag not in extractedfilesdict]
if unknown_tags:
    raise KeyError(f"Tags {unknown_tags} are in merge_composition['{compo_sel}'] but not in extractedfilesdict")

tag_finaldata = compo_sel

## Input - Output files definition

In [ ]:
# Input files: one per tag, in the order given by the composition.
# NB: the order matters when dropping duplicates (the last file wins).
input_files = [Path(extractedfilesdict[tag]) for tag in tag_list]

# Check that all input files exist before reading anything
missing = [str(f) for f in input_files if not f.is_file()]
if missing:
    raise FileNotFoundError(f"Missing input files (current dir = {Path.cwd()}):\n" + "\n".join(missing))

print("list of input files ::")
pprint.pprint([str(f) for f in input_files])

In [ ]:
output_file = Path(mergedextractedfilesdict[compo_sel])

# Create the output directory if needed (np.save does not create missing directories)
output_file.parent.mkdir(parents=True, exist_ok=True)

print("output file ::")
print(output_file.resolve())

## Read input files

In [ ]:
df_spectra_coll = []
for tag, infile in zip(tag_list, input_files):
    # Pickle is needed because the structured arrays may contain object columns
    specdata = np.load(infile, allow_pickle=True)
    df_spec = pd.DataFrame(specdata)
    if ADD_SOURCE_COLUMN:
        df_spec['RUN_TAG'] = tag
    print(f"{tag} :: {infile.name} :: {len(df_spec)} rows, {len(df_spec.columns)} columns")
    display(df_spec.head(n=2))
    df_spectra_coll.append(df_spec)

# All files should have the same columns, otherwise concat silently fills with NaN
ref_columns = set(df_spectra_coll[0].columns)
for tag, df in zip(tag_list, df_spectra_coll):
    diff = ref_columns ^ set(df.columns)
    if diff:
        print(f"WARNING : columns of '{tag}' differ from the first file : {sorted(diff)}")

## Really concatenate

In [ ]:
df_all = pd.concat(df_spectra_coll, ignore_index=True)
n_before = len(df_all)

# Remove exposures present in several runs: keep the last occurrence (i.e. the last file of the composition),
# then sort by observation date.
df_spec_concat = (
    df_all
    .drop_duplicates(subset=KEY_COLUMN, keep='last')
    .sort_values(by=KEY_COLUMN, ascending=True)
    .reset_index(drop=True)
)
print(f"{n_before} rows before, {len(df_spec_concat)} rows after removing {n_before - len(df_spec_concat)} duplicates on '{KEY_COLUMN}'")

In [ ]:
print(f"all data   ==>   {tag_finaldata}  ::  (tmin,tmax) = {df_spec_concat[KEY_COLUMN].min()} , {df_spec_concat[KEY_COLUMN].max()}")
print(f"all data   ==>   {tag_finaldata}  ::  N = {len(df_spec_concat)}")

In [ ]:
display(df_spec_concat.head(n=3))
display(df_spec_concat.tail(n=3))

## Save Data

In [ ]:
# Convert to a numpy structured (record) array, same format as the input files
rec_array = df_spec_concat.to_records(index=False)

# Write to a temporary file first, then rename: an interrupted write cannot corrupt an existing output file.
# (the temporary name must end with '.npy', otherwise np.save appends the extension)
tmp_file = output_file.with_name(output_file.stem + '.tmp.npy')
np.save(tmp_file, rec_array)
tmp_file.replace(output_file)

print(f"saved {len(rec_array)} rows in {output_file.resolve()}")

## Check readability

In [ ]:
# Read the file back and check that it matches what was written
newspecdata = np.load(output_file, allow_pickle=True)
df_newspec = pd.DataFrame(newspecdata)

assert len(df_newspec) == len(df_spec_concat), 'row count mismatch after re-reading the file'
assert list(df_newspec.columns) == list(df_spec_concat.columns), 'column mismatch after re-reading the file'
print('readback OK')

In [ ]:
display(df_newspec.head(n=3))
display(df_newspec.tail(n=3))

In [ ]:
print(f"new data   ==>   {tag_finaldata}  ::  (tmin,tmax) = {df_newspec[KEY_COLUMN].min()} , {df_newspec[KEY_COLUMN].max()}")
print(f"new data   ==>   {tag_finaldata}  ::  N = {len(df_newspec)}")